# ⚡ Energy Consumption Data Preprocessing & Feature Engineering Pipeline
### **Project Code:** EEE-PJ-2026 | **Course:** Research Based Mini Project / Capstone
### **Purpose:** Clean, normalize, and engineer features for Short-Term Energy Load Forecasting (LSTM / GRU / Machine Learning Models).

---

## 📌 Executive Overview for Project Guide & Evaluators

This notebook executes a **simplified, guide-approved 6-step data preprocessing pipeline** on raw energy consumption datasets (Dataset 1: PJM Regional Hourly & Dataset 2: 30-Min Interval Smart Meter Readings).

### 🎯 Key Design Principles:
1. **Guide-Approved Minimal Feature Set:** Removed redundant manual lag columns (`lag_1`, `lag_2`, `rolling_mean`) because **recurrent sequential models (LSTM/GRU) process past temporal context natively** through sliding input windows `(batch_size, sequence_length, features)`.
2. **Min-Max Normalization:** Scales load consumption to $[0.0, 1.0]$ range to prevent exploding gradients and ensure fast neural network convergence.
3. **Cyclical Feature Encoding:** Converts discrete time slots and months into continuous $\sin/\cos$ coordinates.
4. **Memory-Safe Chunked Processing:** Streams data in $1,000,000$-row chunks to run efficiently within $16	ext{ GB}$ RAM systems without memory overflow.
5. **High-Performance Binary Storage:** Outputs compressed Apache Parquet (`.parquet`) binary files for $10	imes$ faster training load times.


## Step 1: Environment Setup & Library Imports
Importing necessary libraries: `pandas` for data manipulation, `numpy` for mathematical cyclical transforms, `json` for saving scale bounds, and `pyarrow` for Parquet export.


In [ ]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# File Configuration
INPUT_CSV = r"V:\Dataset original\CD_INTERVAL_READING_ALL_NO_QUOTES.csv"
OUTPUT_DIR = r"V:\DATASETS\DATASET 2\Preprocessed"
OUTPUT_PARQUET = os.path.join(OUTPUT_DIR, "CD_INTERVAL_simplified_preprocessed.parquet")
SCALER_JSON = os.path.join(OUTPUT_DIR, "scaler_params.json")

CHUNK_SIZE = 1_000_000
USE_COLS = ['CUSTOMER_ID', 'READING_DATETIME', 'GENERAL_SUPPLY_KWH']
DTYPES = {'CUSTOMER_ID': 'int32', 'GENERAL_SUPPLY_KWH': 'float32'}

print("✅ Setup Complete!")
print(f"Input Dataset Path: {INPUT_CSV}")
print(f"Target Output Path: {OUTPUT_PARQUET}")


## Step 2: Global Min-Max Normalization Parameter Estimation
Neural networks require inputs to be scaled between $0.0$ and $1.0$. 
We sample 2,000,000 rows to estimate global $\text{Min}$ ($0.0\text{ kWh}$) and $\text{Max}$ ($99.9\text{th percentile}$ cap) to clip sensor noise spikes before scaling.


In [ ]:
print("[Step 2] Estimating Global Min-Max Scaling Parameters...")

sample_df = pd.read_csv(
    INPUT_CSV, nrows=2_000_000, usecols=USE_COLS, dtype=DTYPES, skipinitialspace=True
)

sample_loads = sample_df['GENERAL_SUPPLY_KWH'].clip(lower=0.0)
min_val = 0.0
max_val = float(sample_loads.quantile(0.999))

if max_val <= 0:
    max_val = 10.0  # Fallback safety cap

scaler_params = {"min": min_val, "max": max_val}
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(SCALER_JSON, 'w') as f:
    json.dump(scaler_params, f, indent=2)

print(f"  • Global Min Load: {min_val:.4f} kWh")
print(f"  • Global Max Load (99.9th percentile cap): {max_val:.4f} kWh")
print(f"  • Saved Scaler Parameters to: {SCALER_JSON}")
del sample_df, sample_loads


## Step 3: Core Cleaning, Normalization & Feature Extraction Function
This function takes each raw data chunk and performs:
1. **Datetime Parsing & Null Dropping:** Converts timestamps to `datetime64` and removes missing entries.
2. **Deduplication:** Drops duplicate readings per customer and timestamp.
3. **Outlier Capping:** Clips negative values to $0.0$ and upper spikes to `max_val`.
4. **Min-Max Scaling:** Scales load to $[0.0, 1.0]$: 
   $$\text{load\_scaled} = \frac{\text{load} - \text{Min}}{\text{Max} - \text{Min}}$$
5. **Guide-Approved Independent Features ($X$):**
   - `step_of_day`: 30-min slot index ($0$ to $47$)
   - `day_of_week`: Day index ($0$ = Mon, $6$ = Sun)
   - `month`: Month index ($1$ to $12$)
   - `is_weekend`: Binary calendar flag ($1$ for Sat/Sun, $0$ for Weekdays)
   - `step_sin`, `step_cos`: Cyclical time transformations
   - `month_sin`, `month_cos`: Cyclical season transformations
6. **Dependent Output Target ($Y$):** `target_load` and `target_scaled` (1-interval ahead shift)


In [ ]:
def process_chunk(chunk, min_val, max_val):
    # 1. Rename to Common Schema
    chunk = chunk.rename(columns={
        'CUSTOMER_ID': 'customer_id',
        'READING_DATETIME': 'datetime',
        'GENERAL_SUPPLY_KWH': 'load_val'
    })
    
    # 2. Datetime Parse & Null Drop
    chunk['datetime'] = pd.to_datetime(chunk['datetime'], errors='coerce')
    chunk = chunk.dropna(subset=['datetime'])
    
    # 3. Deduplicate
    chunk = chunk.drop_duplicates(subset=['customer_id', 'datetime'], keep='first')
    
    # 4. Outlier Handling (Clip 0 to max_val)
    chunk['load_val'] = chunk['load_val'].clip(lower=0.0, upper=max_val)
    
    # 5. Min-Max Normalization [0.0, 1.0]
    chunk['load_scaled'] = ((chunk['load_val'] - min_val) / (max_val - min_val)).astype('float32')
    
    # 6. Feature Extraction
    dt = chunk['datetime']
    hour = dt.dt.hour.astype('int8')
    minute = dt.dt.minute.astype('int8')
    step_of_day = (hour * 2 + (minute // 30)).astype('int8')
    day_of_week = dt.dt.dayofweek.astype('int8')
    month = dt.dt.month.astype('int8')
    is_weekend = day_of_week.isin([5, 6]).astype('int8')
    
    chunk['step_of_day'] = step_of_day
    chunk['day_of_week'] = day_of_week
    chunk['month'] = month
    chunk['is_weekend'] = is_weekend
    
    # Cyclical Transformations
    chunk['step_sin'] = np.sin(2 * np.pi * step_of_day / 48).astype('float32')
    chunk['step_cos'] = np.cos(2 * np.pi * step_of_day / 48).astype('float32')
    chunk['month_sin'] = np.sin(2 * np.pi * month / 12).astype('float32')
    chunk['month_cos'] = np.cos(2 * np.pi * month / 12).astype('float32')
    
    # 7. Dependent Target Creation (Next 30-min load forecast Y)
    chunk['target_load'] = chunk['load_val'].shift(-1).ffill().astype('float32')
    chunk['target_scaled'] = chunk['load_scaled'].shift(-1).ffill().astype('float32')
    
    return chunk.dropna()

print("✅ Chunk Processing Function Defined!")


## Step 4: Chunked Processing & Streaming Parquet Export
We stream through the dataset in chunks of $1,000,000$ rows, applying the processing function and appending directly into a compressed PyArrow `.parquet` file.


In [ ]:
print("[Step 4] Starting Full Dataset Processing Loop...")
start_time = time.time()

reader = pd.read_csv(
    INPUT_CSV,
    chunksize=CHUNK_SIZE,
    usecols=USE_COLS,
    dtype=DTYPES,
    skipinitialspace=True
)

writer = None
total_rows_in = 0
total_rows_out = 0

for i, chunk in enumerate(reader):
    total_rows_in += len(chunk)
    processed_chunk = process_chunk(chunk, min_val, max_val)
    total_rows_out += len(processed_chunk)
    
    # Stream to Parquet Writer
    table = pa.Table.from_pandas(processed_chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PARQUET, table.schema, compression='snappy')
    writer.write_table(table)
    
    if (i + 1) % 50 == 0 or (i + 1) == 345:
        print(f"  • Processed Chunk {i+1}: {total_rows_in:,} rows read so far...")

if writer:
    writer.close()

elapsed = time.time() - start_time
print("
" + "=" * 60)
print("🎉 PREPROCESSING PIPELINE COMPLETE!")
print(f"  • Total Input Rows Read:    {total_rows_in:,}")
print(f"  • Total Clean Output Rows: {total_rows_out:,}")
print(f"  • Processing Time:         {elapsed / 60:.2f} minutes")
print(f"  • Output Parquet Path:     {OUTPUT_PARQUET}")
print("=" * 60)


## Step 5: Verification & Preprocessed Dataset Inspection
We load the final preprocessed Parquet file to verify the column schema, summary statistics, and data types for guide review.


In [ ]:
# Inspect Preprocessed Parquet
pf = pq.ParquetFile(OUTPUT_PARQUET)
print("=== PARQUET DATASET METADATA ===")
print(f"Total Preprocessed Rows: {pf.metadata.num_rows:,}")
print(f"Total Columns:           {pf.metadata.num_columns}")

# Load first 1,000 rows for verification
sample_out = pd.read_parquet(OUTPUT_PARQUET).head(1000)

print("
=== PREPROCESSED DATASET HEAD (First 5 Rows) ===")
display(sample_out.head())

print("
=== COLUMN DATA TYPES & MISSING VALUES ===")
print(sample_out.info())

print("
=== SUMMARY STATISTICS (Scaled Range Check [0, 1]) ===")
display(sample_out[['load_val', 'load_scaled', 'target_val' if 'target_val' in sample_out else 'target_load', 'target_scaled']].describe())
